# Lab 1F-2: Modern Deep-Learning CLIP + FAISS

Exercise 1.F Pipeline 2 is the modern deep-learning retrieval pipeline. It keeps the same Flickr8k multimedia objects and the same shared evaluation protocol as Notebook 1, but replaces explicit handcrafted features with CLIP embeddings.

CLIP maps images and text into a shared vector space. An embedding is a learned vector representation, so a text query and an image can be compared directly by similarity. FAISS stores the image vectors in a vector index and retrieves nearest neighbors efficiently.

This remains an MDM notebook: represent media, store descriptors, index/search descriptors, rank results, and evaluate retrieval. Compare results carefully because stronger semantic matching depends on the query type and on how relevance is defined.

### Colab dependency check

This notebook uses FAISS for vector search. FAISS is often not preinstalled in a fresh Colab runtime, so the next cell checks whether required packages are available and installs only the missing ones.

If an import error persists after installation, restart the runtime and run the notebook from the top.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = [
    ("datasets", "datasets"),
    ("transformers", "transformers"),
    ("faiss-cpu", "faiss"),
]

missing_packages = [
    package_name
    for package_name, import_name in REQUIRED_PACKAGES
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing_packages,
    ])
    print("Installation complete. Continue by running the setup cell below.")
else:
    print("All required packages are already available.")

## 0. Setup

`FAST_DEV` is a smoke-test mode. It keeps Colab and classroom execution fast by using a small image subset, but it must not change the retrieval pipeline, query modes, or evaluation protocol. FAST_DEV results are useful for checking that the notebook runs; they are not final benchmark numbers.

Set `FAST_DEV=False` for the full Flickr8k run before using the results in any final comparison or slide narrative.

In [ ]:
FAST_DEV = True
FAST_DEV_LIMIT = 200

from pathlib import Path
import hashlib
import json
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from transformers import CLIPModel, CLIPProcessor

try:
    import faiss
except ImportError as exc:
    raise ImportError('Install faiss-cpu before running this notebook.') from exc

LAB_DIR = Path('content/lab/1F')
if not (LAB_DIR / 'evaluation_queries.json').exists():
    LAB_DIR = Path('.')
OUTPUT_DIR = LAB_DIR / 'outputs'
FIGURE_DIR = OUTPUT_DIR / 'figures'
QUERY_FILE = LAB_DIR / 'evaluation_queries.json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

with QUERY_FILE.open('r', encoding='utf-8') as f:
    evaluation_queries = json.load(f)

reference_image_ids = {
    query['reference_image_id']
    for query in evaluation_queries
    if query.get('reference_image_id')
    and {'image', 'fused'} & set(query.get('query_modes', []))
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'FAST_DEV={FAST_DEV}, device={device}')
print(f'Evaluation protocol contains {len(evaluation_queries)} queries.')
print(f'Image/fused queries require {len(reference_image_ids)} reference images.')

## 1. Load Flickr8k and Build Stable Image IDs

Flickr8k contains images and captions. In this lab, each image is treated as one multimedia object: the image is unstructured visual media, and the captions are textual descriptions attached to that object.

Stable image IDs are important because Notebook 1 and Notebook 2 must refer to the same objects when they use `evaluation_queries.json`. In the Part E architecture, this section corresponds to the media/content DB and the structured metadata used to identify records.

In [ ]:
from datasets import load_dataset, concatenate_datasets

print('Loading Flickr8k...')
ds = load_dataset('jxie/flickr8k')
split_sizes = pd.DataFrame({
    'split': list(ds.keys()),
    'rows': [len(ds[name]) for name in ds.keys()],
})
display(split_sizes)

source_ds = concatenate_datasets([ds['train'], ds['test']])
full_ds = source_ds
print(f'Images available in the source collection: {len(source_ds)}')


def normalize_text(text):
    return re.sub(r'[^a-z0-9]+', ' ', str(text).lower()).strip()


def caption_list(item):
    caps = item.get('caption', [])
    if isinstance(caps, str):
        caps = [caps]
    elif not caps:
        caps = [item.get(f'caption_{i}', '') for i in range(5)]
    cleaned = []
    for c in caps:
        if isinstance(c, dict):
            c = c.get('raw') or c.get('text') or ''
        s = str(c).strip()
        if s:
            cleaned.append(s)
    return cleaned


def stable_image_id(item, idx, image):
    for key in ('image_id', 'img_id', 'filename', 'file_name'):
        value = item.get(key)
        if value:
            return Path(str(value)).name
    filename = getattr(image, 'filename', '')
    if filename:
        return Path(filename).name
    captions = caption_list(item)
    first_caption = captions[0] if captions else ''
    w, h = image.size
    digest = hashlib.sha1(f'{first_caption}|{w}x{h}'.encode('utf-8')).hexdigest()[:12]
    return f'flickr8k_{digest}'


records = []
for idx in range(len(source_ds)):
    item = source_ds[idx]
    image = item['image'].convert('RGB')
    w, h = image.size
    captions = caption_list(item)
    records.append({
        'image_id': stable_image_id(item, idx, image),
        'dataset_pos': idx,
        'width': w,
        'height': h,
        'captions': captions,
        'all_captions': ' '.join(captions),
    })

full_df = pd.DataFrame(records)
dup_count = int(full_df['image_id'].duplicated().sum())
if dup_count:
    warnings.warn(f'Found {dup_count} duplicate image rows; keeping first occurrence per image_id.')
    full_df = full_df.drop_duplicates(subset='image_id', keep='first').reset_index(drop=True)

if FAST_DEV:
    reference_df = full_df[full_df['image_id'].isin(reference_image_ids)].copy()
    missing_refs = sorted(reference_image_ids - set(reference_df['image_id']))

    if missing_refs:
        raise ValueError(
            'FAST_DEV cannot run because some required reference images are missing from Flickr8k: '
            f'{missing_refs}. Check evaluation_queries.json and stable_image_id generation.'
        )

    if len(reference_df) > FAST_DEV_LIMIT:
        raise ValueError(
            f'FAST_DEV_LIMIT={FAST_DEV_LIMIT} is smaller than the number of required reference images '
            f'({len(reference_df)}). Increase FAST_DEV_LIMIT.'
        )

    remaining_df = full_df[~full_df['image_id'].isin(reference_image_ids)].copy()
    n_remaining = FAST_DEV_LIMIT - len(reference_df)
    df = pd.concat([reference_df, remaining_df.head(n_remaining)], ignore_index=True)

    print('FAST_DEV is active.')
    print(f'Subset size: {len(df)} images.')
    print(f'Required reference images included: {len(reference_df)} / {len(reference_image_ids)}.')
else:
    df = full_df.reset_index(drop=True)
    print(f'FULL mode is active. Dataset size: {len(df)} images.')

image_id_to_pos = dict(zip(df['image_id'], df['dataset_pos']))
ordered_image_ids = df['image_id'].tolist()

fast_dev_summary = pd.DataFrame([
    {
        'mode': 'FAST_DEV' if FAST_DEV else 'FULL',
        'indexed_images': len(df),
        'evaluation_queries': len(evaluation_queries),
        'required_reference_images': len(reference_image_ids),
        'reference_images_present': len(reference_image_ids & set(df['image_id'])),
    }
])
display(fast_dev_summary)

signature = {
    'fast_dev': FAST_DEV,
    'count': int(len(df)),
    'first_items': df[['image_id', 'width', 'height', 'captions']].head(5).to_dict(orient='records'),
}
signature_path = OUTPUT_DIR / ('dataset_signature_clip_fast_dev.json' if FAST_DEV else 'dataset_signature_clip_full.json')
if signature_path.exists():
    old_signature = json.loads(signature_path.read_text(encoding='utf-8'))
    if old_signature.get('first_items') != signature['first_items']:
        warnings.warn(f'Dataset signature changed: {signature_path}')
signature_path.write_text(json.dumps(signature, indent=2), encoding='utf-8')

print('Dataframe columns:')
display(pd.DataFrame({'column': df.columns}))
display(df[['image_id', 'captions']].head(5))

sample_row = df.iloc[0]
sample_image = full_ds[int(sample_row['dataset_pos'])]['image'].convert('RGB')
plt.figure(figsize=(4, 3))
plt.imshow(sample_image)
plt.axis('off')
plt.show()
print(f"Sample image_id: {sample_row['image_id']}")
for idx, caption in enumerate(sample_row['captions'][:5], start=1):
    print(f'{idx}. {caption}')

## 2. Load CLIP ViT-B/32

CLIP is a pretrained image-text encoder. It has an image encoder and a text encoder, and both produce vectors in the same embedding space. Similarity is computed between normalized vectors, so an inner product behaves like cosine similarity.

This is the intended modern contrast with Notebook 1: the representation is learned from image-text training data, not built from explicit HSV histograms, edge density, or lexical TF-IDF features. The visual encoder divides images into fixed-size patches internally as part of representation learning; this is not the image-segmentation module from the Part E architecture.

In [ ]:
MODEL_NAME = 'openai/clip-vit-base-patch32'

print(f'Loading {MODEL_NAME}...')
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()
EMBED_DIM = int(model.config.projection_dim)
print(f'Embedding dimension: {EMBED_DIM}')

## 3. Encode Images with Caching

This section builds the descriptor table for the visual media collection. Each image becomes one CLIP embedding, and each row of `image_embeddings` is aligned with one `image_id` in `ordered_image_ids`.

The cache avoids recomputing embeddings during repeated classroom runs. The cache key distinguishes FAST_DEV from the full run and preserves the stable image-id order, so the vector rows still map back to the correct media objects.

In [ ]:
cache_tag = 'fast_dev' if FAST_DEV else 'full'
embedding_path = OUTPUT_DIR / f'clip_image_embeddings_{cache_tag}.npy'
ids_path = OUTPUT_DIR / f'clip_image_ids_{cache_tag}.json'


def encode_image_batch(images):
    inputs = processor(images=images, return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        embeddings = model.get_image_features(**inputs)
    if not torch.is_tensor(embeddings):
        if hasattr(embeddings, 'image_embeds') and embeddings.image_embeds is not None:
            embeddings = embeddings.image_embeds
        elif hasattr(embeddings, 'pooler_output') and embeddings.pooler_output is not None:
            embeddings = embeddings.pooler_output
        elif hasattr(embeddings, 'last_hidden_state'):
            embeddings = embeddings.last_hidden_state[:, 0, :]
        else:
            raise TypeError(f'Unsupported CLIP embedding output type: {type(embeddings)}')
    embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return embeddings.cpu().numpy().astype('float32')


def encode_all_images(batch_size=32):
    batches = []
    for start in range(0, len(df), batch_size):
        end = min(start + batch_size, len(df))
        images = [full_ds[int(df.iloc[i]['dataset_pos'])]['image'].convert('RGB') for i in range(start, end)]
        batches.append(encode_image_batch(images))
        print(f'Encoded {end}/{len(df)} images')
    print()
    return np.vstack(batches).astype('float32')


if embedding_path.exists() and ids_path.exists():
    cached_ids = json.loads(ids_path.read_text(encoding='utf-8'))
    if cached_ids == ordered_image_ids:
        image_embeddings = np.load(embedding_path).astype('float32')
        print(f'Loaded cached embeddings from {embedding_path}')
    else:
        warnings.warn('Cached image-id order changed; recomputing embeddings.')
        image_embeddings = encode_all_images()
        np.save(embedding_path, image_embeddings)
        ids_path.write_text(json.dumps(ordered_image_ids, indent=2), encoding='utf-8')
else:
    image_embeddings = encode_all_images()
    np.save(embedding_path, image_embeddings)
    ids_path.write_text(json.dumps(ordered_image_ids, indent=2), encoding='utf-8')

print(f'image_embeddings shape: {image_embeddings.shape}')

## 4. FAISS IndexFlatIP Main Result

FAISS is a vector index: a data structure for nearest-neighbor search over embeddings. `IndexFlatIP` performs exact inner-product search, not approximate search. Because the CLIP embeddings are normalized, inner product behaves like cosine similarity.

In the Part E architecture, the embeddings are the feature/descriptor DB, and the FAISS search step is part of the index-based query processor. Approximate nearest-neighbor indexing is the scalable idea for much larger collections, but this notebook's main Flickr8k result uses exact `IndexFlatIP`.

In [ ]:
index_flat = faiss.IndexFlatIP(image_embeddings.shape[1])
index_flat.add(image_embeddings)
print(f'IndexFlatIP vectors: {index_flat.ntotal}')

## 5. Modern Retrieval Functions

These functions turn user queries into query embeddings and rank database items by similarity score. A text query uses the CLIP text encoder. A visual reference query uses the CLIP image encoder. Fused retrieval combines textual and visual evidence by averaging normalized text and image query embeddings.

Higher scores mean stronger similarity in the learned CLIP space. Semantic matches may differ from lexical matches because CLIP compares learned cross-modal meaning, not exact caption terms. The ranked outputs should still be inspected as retrieval results, not treated as automatically correct labels.

In [ ]:
def encode_text_query(query_text):
    inputs = processor(text=[query_text], return_tensors='pt', padding=True, truncation=True, max_length=77).to(device)
    with torch.no_grad():
        embedding = model.get_text_features(**inputs)
    if not torch.is_tensor(embedding):
        if hasattr(embedding, 'text_embeds') and embedding.text_embeds is not None:
            embedding = embedding.text_embeds
        elif hasattr(embedding, 'pooler_output') and embedding.pooler_output is not None:
            embedding = embedding.pooler_output
        elif hasattr(embedding, 'last_hidden_state'):
            embedding = embedding.last_hidden_state[:, 0, :]
        else:
            raise TypeError(f'Unsupported CLIP text output type: {type(embedding)}')
    embedding = embedding / embedding.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return embedding.cpu().numpy().astype('float32')


def encode_image_query(query_image):
    return encode_image_batch([query_image.convert('RGB')])


def search_index(query_embedding, exclude_image_id=None, top_k=10):
    search_k = min(top_k + (1 if exclude_image_id else 0), len(ordered_image_ids))
    scores, indices = index_flat.search(query_embedding.astype('float32'), search_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        image_id = ordered_image_ids[int(idx)]
        if exclude_image_id is not None and image_id == exclude_image_id:
            continue
        results.append({'image_id': image_id, 'score': float(score), 'rank': len(results) + 1})
        if len(results) >= top_k:
            break
    return results


def modern_text_to_image(query_text, exclude_image_id=None, top_k=10):
    return search_index(encode_text_query(query_text), exclude_image_id=exclude_image_id, top_k=top_k)


def modern_image_to_image(query_image, exclude_image_id=None, top_k=10):
    return search_index(encode_image_query(query_image), exclude_image_id=exclude_image_id, top_k=top_k)


def modern_fused_retrieval(query_text, query_image, text_weight=0.5, exclude_image_id=None, top_k=10):
    text_emb = encode_text_query(query_text)
    image_emb = encode_image_query(query_image)
    fused = text_weight * text_emb + (1.0 - text_weight) * image_emb
    fused = fused / np.maximum(np.linalg.norm(fused, axis=1, keepdims=True), 1e-12)
    return search_index(fused.astype('float32'), exclude_image_id=exclude_image_id, top_k=top_k)

## 6. Shared Evaluation Protocol

`evaluation_queries.json` is the shared evaluation protocol used by Notebook 1 and Notebook 2. It contains `query_id`, `query_text`, `reference_image_id`, `required_terms`, `optional_terms`, `query_modes`, and optional `manual_relevant_image_ids`.

The shared file exists so both pipelines are evaluated on the same query set. Text queries use `query_text`. Image and fused queries require a real `reference_image_id` that is present in the indexed dataframe. If manual relevance labels are empty, relevance is derived from required caption terms.

The validation below loads and checks the protocol; it does not rewrite the JSON or change query modes.

In [ ]:
with QUERY_FILE.open('r', encoding='utf-8') as f:
    evaluation_queries = json.load(f)

evaluation_df = pd.DataFrame(evaluation_queries)
protocol_columns = [
    'query_id',
    'query_text',
    'reference_image_id',
    'required_terms',
    'optional_terms',
    'query_modes',
    'manual_relevant_image_ids',
]
display(evaluation_df[protocol_columns])

text_query_embeddings = np.vstack([
    encode_text_query(query['query_text'])
    for query in evaluation_queries
    if query.get('query_text')
]).astype('float32')
embedding_shapes = pd.DataFrame([
    {
        'matrix': 'image_embeddings',
        'rows': int(image_embeddings.shape[0]),
        'columns': int(image_embeddings.shape[1]),
        'row_meaning': 'one indexed image',
    },
    {
        'matrix': 'text_query_embeddings',
        'rows': int(text_query_embeddings.shape[0]),
        'columns': int(text_query_embeddings.shape[1]),
        'row_meaning': 'one text query',
    },
])
display(embedding_shapes)

indexed_ids = set(df['image_id'])
missing_reference_rows = []
for query in evaluation_queries:
    modes = set(query.get('query_modes') or [])
    if modes & {'image', 'fused'}:
        reference_id = query.get('reference_image_id')
        if not reference_id or reference_id not in indexed_ids:
            missing_reference_rows.append({
                'query_id': query.get('query_id'),
                'reference_image_id': reference_id,
                'query_modes': sorted(modes),
            })

if missing_reference_rows:
    missing_df = pd.DataFrame(missing_reference_rows)
    display(missing_df)
    raise ValueError(
        'Image and fused queries require reference_image_id values that exist in the indexed dataframe. '
        'Update evaluation_queries.json with real indexed Flickr8k image IDs before running those modes.'
    )


def term_in_caption(term, caption):
    return normalize_text(term) in normalize_text(caption)


def captions_match_required(captions, required_terms):
    return all(any(term_in_caption(term, caption) for caption in captions) for term in required_terms)


def relevance_ids(query):
    relevant = set(query.get('manual_relevant_image_ids') or [])
    required_terms = query.get('required_terms') or []
    for _, row in df.iterrows():
        if captions_match_required(row['captions'], required_terms):
            relevant.add(row['image_id'])
    relevant.discard(query.get('reference_image_id'))
    return relevant


def reference_image_for_query(query):
    reference_id = query['reference_image_id']
    return reference_id, full_ds[int(image_id_to_pos[reference_id])]['image']


def queries_for_mode(mode):
    return [q for q in evaluation_queries if mode in q.get('query_modes', [])]


K_VALUES = [1, 5, 10, 20, 50]


def precision_at_k(retrieved_ids, relevant, k):
    return len(set(retrieved_ids[:k]) & relevant) / k if k else 0.0


def recall_at_k(retrieved_ids, relevant, k):
    return len(set(retrieved_ids[:k]) & relevant) / len(relevant) if relevant else 0.0


def summarize_metrics(per_query):
    summary = {}
    for k in K_VALUES:
        summary[str(k)] = {
            'avg_precision': float(np.mean([row[f'p@{k}'] for row in per_query])) if per_query else 0.0,
            'avg_recall': float(np.mean([row[f'r@{k}'] for row in per_query])) if per_query else 0.0,
        }
    return summary

### Evaluation Metrics

Precision@K asks: among the top-K retrieved images, how many are relevant? Recall@K asks: among all relevant images, how many were recovered in the top-K?

Relevance here is simple and inspectable. It uses manual labels when provided; otherwise it is derived from required caption terms. This is not a perfect human relevance judgment. FAST_DEV metrics are sanity checks only, not final benchmark numbers.

In [ ]:
def evaluate_modern_text(text_queries, top_k=max(K_VALUES)):
    per_query = []
    for query in text_queries:
        relevant = relevance_ids(query)
        retrieved = modern_text_to_image(query['query_text'], exclude_image_id=query.get('reference_image_id'), top_k=top_k)
        retrieved_ids = [row['image_id'] for row in retrieved]
        row = {'query_id': query['query_id'], 'relevant_count': len(relevant), 'retrieved_ids': retrieved_ids}
        for k in K_VALUES:
            row[f'p@{k}'] = precision_at_k(retrieved_ids, relevant, k)
            row[f'r@{k}'] = recall_at_k(retrieved_ids, relevant, k)
        per_query.append(row)
    return {'per_query': per_query, 'summary': summarize_metrics(per_query)}


def evaluate_modern_image(image_queries, top_k=max(K_VALUES)):
    per_query = []
    for query in image_queries:
        relevant = relevance_ids(query)
        reference_id, query_image = reference_image_for_query(query)
        retrieved = modern_image_to_image(query_image, exclude_image_id=reference_id, top_k=top_k)
        retrieved_ids = [row['image_id'] for row in retrieved]
        row = {'query_id': query['query_id'], 'reference_image_id': reference_id, 'relevant_count': len(relevant), 'retrieved_ids': retrieved_ids}
        for k in K_VALUES:
            row[f'p@{k}'] = precision_at_k(retrieved_ids, relevant, k)
            row[f'r@{k}'] = recall_at_k(retrieved_ids, relevant, k)
        per_query.append(row)
    return {'per_query': per_query, 'summary': summarize_metrics(per_query)}


def evaluate_modern_fused(dual_queries, text_weight=0.5, top_k=max(K_VALUES)):
    per_query = []
    for query in dual_queries:
        relevant = relevance_ids(query)
        reference_id, query_image = reference_image_for_query(query)
        retrieved = modern_fused_retrieval(query['query_text'], query_image, text_weight=text_weight, exclude_image_id=reference_id, top_k=top_k)
        retrieved_ids = [row['image_id'] for row in retrieved]
        row = {'query_id': query['query_id'], 'reference_image_id': reference_id, 'relevant_count': len(relevant), 'retrieved_ids': retrieved_ids}
        for k in K_VALUES:
            row[f'p@{k}'] = precision_at_k(retrieved_ids, relevant, k)
            row[f'r@{k}'] = recall_at_k(retrieved_ids, relevant, k)
        per_query.append(row)
    return {'per_query': per_query, 'summary': summarize_metrics(per_query)}


modern_text = evaluate_modern_text(queries_for_mode('text'))
modern_image = evaluate_modern_image(queries_for_mode('image'))
modern_fused = evaluate_modern_fused(queries_for_mode('fused'))

modern_results = {
    'fast_dev': FAST_DEV,
    'model_name': MODEL_NAME,
    'faiss_index': 'IndexFlatIP',
    'k_values': K_VALUES,
    'text_to_image': modern_text,
    'image_to_image': modern_image,
    'fused': modern_fused,
}

summary_rows = []
for mode_name, result in [
    ('text_to_image', modern_text),
    ('image_to_image', modern_image),
    ('fused', modern_fused),
]:
    for k in K_VALUES:
        summary_rows.append({
            'mode': mode_name,
            'k': k,
            'avg_precision': result['summary'][str(k)]['avg_precision'],
            'avg_recall': result['summary'][str(k)]['avg_recall'],
        })
summary_table = pd.DataFrame(summary_rows)
display(summary_table)

The table above summarizes retrieval quality for each query mode. Higher Precision@K means the top of the ranking contains more relevant images. Higher Recall@K means the ranking recovers more of the relevant set.

For FAST_DEV, look only for sanity signals: plausible nonzero values, no broken query mode, and differences from Notebook 1 that are worth inspecting. Do not claim final superiority from a FAST_DEV subset.

## 7. Classic vs Modern Comparison

This section overlays modern results with the saved Notebook 1 output when it is available. The goal is to compare representation choices: Notebook 1 ranks explicit lexical and handcrafted visual descriptors, while Notebook 2 ranks learned multimodal embeddings.

Use the plot as a diagnostic view. It can show whether outputs differ from Notebook 1, but FAST_DEV is still only a smoke test.

In [ ]:
classic_path = OUTPUT_DIR / ('classic_results_fast_dev.json' if FAST_DEV else 'classic_results_full.json')
classic_results = json.loads(classic_path.read_text(encoding='utf-8')) if classic_path.exists() else None
if classic_results is None:
    print(f'Classic results not found at {classic_path}; run Notebook 1 first for direct overlay.')
else:
    print(f'Loaded classic comparison data from {classic_path}')


def plot_comparison(classic_results, modern_results):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    series = []
    if classic_results is not None:
        series.extend([
            ('classic text-only', classic_results['text_only']['summary']),
            ('classic visual-only', classic_results['visual_only']['summary']),
            ('classic fused', classic_results['fused']['summary']),
        ])
    series.extend([
        ('modern text-to-image', modern_results['text_to_image']['summary']),
        ('modern image-to-image', modern_results['image_to_image']['summary']),
        ('modern fused', modern_results['fused']['summary']),
    ])
    for label, summary in series:
        axes[0].plot(K_VALUES, [summary[str(k)]['avg_precision'] for k in K_VALUES], marker='o', label=label)
        axes[1].plot(K_VALUES, [summary[str(k)]['avg_recall'] for k in K_VALUES], marker='o', label=label)
    axes[0].set_title('Average Precision@K')
    axes[1].set_title('Average Recall@K')
    for ax in axes:
        ax.set_xlabel('K')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


fig = plot_comparison(classic_results, modern_results)
if not FAST_DEV:
    fig.savefig(FIGURE_DIR / 'classic_modern_clip_comparison.png', dpi=150)
plt.show()

The plot compares average Precision@K and Recall@K across modes. Read it together with retrieved examples: a higher curve is useful only if the retrieved images are plausible matches for the query.

Semantic CLIP matches can improve some text queries, visual reference queries can retrieve images with similar visual content, and fused retrieval combines both signals. The tradeoff is lower interpretability than explicit color, edge, or TF-IDF features.

## 8. Optional Scalability / Latency Section

`IndexFlatIP` remains the main Flickr8k result and is exact. The IVF/nprobe experiment below is optional and only illustrates the latency/recall tradeoff that appears in larger vector collections.

This maps to the scalable query-processing part of the Part E architecture: the descriptors are the same kind of media representation, but the index structure changes how fast nearest-neighbor search can run.

In [ ]:
if len(image_embeddings) >= 100:
    nlist = min(50, max(2, int(np.sqrt(len(image_embeddings)))))
    quantizer = faiss.IndexFlatIP(image_embeddings.shape[1])
    index_ivf = faiss.IndexIVFFlat(quantizer, image_embeddings.shape[1], nlist, faiss.METRIC_INNER_PRODUCT)
    index_ivf.train(image_embeddings)
    index_ivf.add(image_embeddings)
    nprobe_values = sorted(set([1, min(2, nlist), min(5, nlist), min(10, nlist)]))
    latency_rows = []
    sample_query = image_embeddings[0:1]
    for nprobe in nprobe_values:
        index_ivf.nprobe = nprobe
        start = time.time()
        for _ in range(20):
            index_ivf.search(sample_query, 10)
        latency_rows.append({'nprobe': int(nprobe), 'latency_ms': (time.time() - start) / 20 * 1000})
    print(pd.DataFrame(latency_rows))
else:
    print('Skipping optional IVF demo: not enough images in this run.')

## 9. MDM Comparison Takeaway

Notebook 1 uses explicit features and classical retrieval: captions become lexical vectors, and images become handcrafted visual descriptors. Notebook 2 uses learned multimodal embeddings and vector indexing.

The MDM concepts remain the same: represent media, store descriptors, index and search descriptors, formulate queries, rank results, and evaluate retrieval. The important lesson is not only that CLIP may be better; it is how representation choice changes retrieval behavior, scalability, interpretability, and failure modes.

## 10. Save Results

FAST_DEV outputs are debugging artifacts. Full-run outputs are the ones to inspect before creating any final slide narrative.

In [ ]:
result_path = OUTPUT_DIR / ('modern_clip_results_fast_dev.json' if FAST_DEV else 'modern_clip_results_full.json')
result_path.write_text(json.dumps(modern_results, indent=2), encoding='utf-8')
print(f'Saved results to {result_path}')